## Modul 05 - Representasi Pengetahuan dan Sistem Pakar

#### Kegiatan 1 — Menyusun Basis Pengetahuan (±50 menit)

In [18]:
aturan = [
    {
        "jika": ["mati total", "lampu indikator mati"],
        "maka": "masalah pada daya",
    },
    {
        "jika": ["masalah pada daya", "baterai kembung"],
        "maka": "DIAGNOSA: baterai rusak, segera ganti",
    },
    {
        "jika": ["masalah pada daya", "charger tidak hangat"],
        "maka": "DIAGNOSA: charger/adaptor rusak",
    },
    {
        "jika": ["menyala", "layar gelap"],
        "maka": "masalah pada layar",
    },
    {
        "jika": ["masalah pada layar", "normal di monitor eksternal"],
        "maka": "DIAGNOSA: kabel fleksibel/panel LCD rusak",
    },
    {
        "jika": ["menyala", "sangat lambat"],
        "maka": "masalah kinerja",
    },
    {
        "jika": ["masalah kinerja", "badan laptop panas"],
        "maka": "DIAGNOSA: overheating, bersihkan kipas & ganti pasta",
    },
    {
        "jika": ["masalah kinerja", "hang saat banyak aplikasi"],
        "maka": "DIAGNOSA: RAM kurang, lakukan upgrade",
    },
]

print("Jumlah aturan:", len(aturan))

Jumlah aturan: 8


Pengembangan 1

In [9]:
semua_kondisi = {kondisi for r in aturan for kondisi in r["jika"]}
kesimpulan_antara = {r["maka"] for r in aturan}
daftar_gejala = sorted(list(semua_kondisi - kesimpulan_antara))

# 2. Tampilkan pilihan ke pengguna
print("=== Sistem Diagnosa Kerusakan Laptop ===")
print("Pilih gejala yang dialami (pisahkan nomor dengan koma, misal: 1, 3):")
for i, g in enumerate(daftar_gejala, 1):
    print(f"{i}. {g}")

pilihan = input("\nNomor gejala: ").split(",")
fakta = {
    daftar_gejala[int(idx.strip()) - 1]
    for idx in pilihan
    if idx.strip().isdigit() and 0 < int(idx.strip()) <= len(daftar_gejala)
}

ada_fakta_baru = True
while ada_fakta_baru:
    ada_fakta_baru = False
    for r in aturan:
        if all(k in fakta for k in r["jika"]) and r["maka"] not in fakta:
            fakta.add(r["maka"])
            ada_fakta_baru = True

hasil = [f for f in fakta if f.startswith("DIAGNOSA:")]

print("\n--- Hasil Analisis ---")
if hasil:
    for h in hasil:
        print(f"-> {h}")
else:
    print("Gejala belum cukup untuk menghasilkan diagnosa pasti.")

=== Sistem Diagnosa Kerusakan Laptop ===
Pilih gejala yang dialami (pisahkan nomor dengan koma, misal: 1, 3):
1. badan laptop panas
2. baterai kembung
3. charger tidak hangat
4. hang saat banyak aplikasi
5. lampu indikator mati
6. layar gelap
7. mati total
8. menyala
9. normal di monitor eksternal
10. sangat lambat

--- Hasil Analisis ---
Gejala belum cukup untuk menghasilkan diagnosa pasti.


#### Kegiatan 2 — Mesin Inferensi: Forward Chaining (±80 menit)

In [11]:
def forward_chaining(aturan, fakta_awal):
    fakta = list(fakta_awal) 
    ada_baru = True

    while ada_baru: 
        ada_baru = False
        for r in aturan:
            terpenuhi = all(k in fakta for k in r["jika"])
            if terpenuhi and r["maka"] not in fakta:
                fakta.append(r["maka"])
                print(" Terpicu:", r["jika"], "->", r["maka"])
                ada_baru = True 

    return fakta

gejala = ["mati total", "lampu indikator mati", "baterai kembung"]
hasil = forward_chaining(aturan, gejala)
print("\nKesimpulan akhir:", [f for f in hasil if f.startswith("DIAGNOSA")])

 Terpicu: ['mati total', 'lampu indikator mati'] -> masalah pada daya
 Terpicu: ['masalah pada daya', 'baterai kembung'] -> DIAGNOSA: baterai rusak, segera ganti

Kesimpulan akhir: ['DIAGNOSA: baterai rusak, segera ganti']


Pengembangan 2

In [13]:
def forward_chaining(aturan, fakta_awal, prefix_target="DIAGNOSA"):
    fakta = set(fakta_awal)
    jejak_inferensi = []
    iterasi = 1
    ada_baru = True

    print(f"[*] Fakta awal yang diketahui: {list(fakta)}")
    print("=" * 60)

    while ada_baru:
        ada_baru = False
        print(f"--- Iterasi ke-{iterasi} ---")

        for r in aturan:
            kondisi = r["jika"]
            kesimpulan = r["maka"]

            if all(k in fakta for k in kondisi) and kesimpulan not in fakta:
                fakta.add(kesimpulan)
                ada_baru = True

                log_teks = f"Premis {kondisi} terpenuhi -> Menghasilkan fakta: '{kesimpulan}'"
                jejak_inferensi.append(
                    {"aturan": r, "langkah": iterasi, "log": log_teks}
                )
                print(f"  [+] Terpicu: {log_teks}")

        if not ada_baru:
            print("  (Tidak ada aturan baru yang terpicu)")

        iterasi += 1

    diagnosa = [f for f in fakta if f.startswith(prefix_target)]
    fakta_perantara = [
        f for f in fakta if f not in fakta_awal and not f.startswith(prefix_target)
    ]

    return {
        "semua_fakta": fakta,
        "fakta_perantara": fakta_perantara,
        "diagnosa": diagnosa,
        "riwayat": jejak_inferensi,
    }

gejala = ["mati total", "lampu indikator mati", "baterai kembung"]
hasil = forward_chaining(aturan, gejala)

print("\n" + "=" * 60)
print(f"Fakta Perantara : {hasil['fakta_perantara']}")
print(f"Kesimpulan Akhir: {hasil['diagnosa']}")

[*] Fakta awal yang diketahui: ['baterai kembung', 'lampu indikator mati', 'mati total']
--- Iterasi ke-1 ---
  [+] Terpicu: Premis ['mati total', 'lampu indikator mati'] terpenuhi -> Menghasilkan fakta: 'masalah pada daya'
  [+] Terpicu: Premis ['masalah pada daya', 'baterai kembung'] terpenuhi -> Menghasilkan fakta: 'DIAGNOSA: baterai rusak, segera ganti'
--- Iterasi ke-2 ---
  (Tidak ada aturan baru yang terpicu)

Fakta Perantara : ['masalah pada daya']
Kesimpulan Akhir: ['DIAGNOSA: baterai rusak, segera ganti']


#### Kegiatan 3 — Backward Chaining: Membuktikan Dugaan

In [14]:
def backward_chaining(aturan, fakta, tujuan, level=0):
    spasi = "  " * level 

    if tujuan in fakta:
        print(spasi + f"'{tujuan}' ada di fakta [OK]")
        return True

    for r in aturan:
        if r["maka"] == tujuan:
            print(spasi + f"Cek aturan: {r['jika']} -> {tujuan}")
            if all(
                backward_chaining(aturan, fakta, k, level + 1)
                for k in r["jika"]
            ):
                return True
    return False

fakta = ["mati total", "lampu indikator mati", "baterai kembung"]
dugaan = "DIAGNOSA: baterai rusak, segera ganti"

print(f"Menguji hipotesis: '{dugaan}'\n")
terbukti = backward_chaining(aturan, fakta, dugaan)
print(f"\nTerbukti? {terbukti}")

Menguji hipotesis: 'DIAGNOSA: baterai rusak, segera ganti'

Cek aturan: ['masalah pada daya', 'baterai kembung'] -> DIAGNOSA: baterai rusak, segera ganti
  Cek aturan: ['mati total', 'lampu indikator mati'] -> masalah pada daya
    'mati total' ada di fakta [OK]
    'lampu indikator mati' ada di fakta [OK]
  'baterai kembung' ada di fakta [OK]

Terbukti? True


Pengembangan 3

In [15]:
def backward_chaining(aturan, fakta, tujuan, level=0):
    spasi = "  " * level
    if tujuan in fakta:
        print(spasi + f"[OK] '{tujuan}' ada di fakta")
        return True

    aturan_cocok = [r for r in aturan if r["maka"] == tujuan]
    if aturan_cocok:
        for r in aturan_cocok:
            print(spasi + f"Cek aturan: {r['jika']} -> {tujuan}")
            if all(
                backward_chaining(aturan, fakta, k, level + 1)
                for k in r["jika"]
            ):
                fakta.append(tujuan)
                return True
        return False

    jawab = input(spasi + f"> Apakah laptop '{tujuan}'? (y/n): ").strip().lower()
    if jawab == "y":
        fakta.append(tujuan)
        return True

    return False


fakta = ["mati total"]
dugaan = "DIAGNOSA: charger/adaptor rusak"

print(f"Menguji hipotesis: '{dugaan}'\n")
if backward_chaining(aturan, fakta, dugaan):
    print(f"\nHasil: {dugaan} (TERBUKTI)")
else:
    print(f"\nHasil: Hipotesis tidak terbukti.")

Menguji hipotesis: 'DIAGNOSA: charger/adaptor rusak'

Cek aturan: ['masalah pada daya', 'charger tidak hangat'] -> DIAGNOSA: charger/adaptor rusak
  Cek aturan: ['mati total', 'lampu indikator mati'] -> masalah pada daya
    [OK] 'mati total' ada di fakta

Hasil: Hipotesis tidak terbukti.


#### Kegiatan 4 — Sistem Pakar Interaktif

In [8]:
gejala_dikenal = [
    "mati total",
    "lampu indikator mati",
    "baterai kembung",
    "charger tidak hangat",
    "menyala",
    "layar gelap",
    "normal di monitor eksternal",
    "sangat lambat",
    "badan laptop panas",
    "hang saat banyak aplikasi",
]

print("=== SISTEM PAKAR DIAGNOSA LAPTOP ===")
fakta_user = []
for g in gejala_dikenal:
    jawab = input(f"Apakah laptop Anda '{g}'? (y/t) ")
    if jawab.lower().startswith("y"):
        fakta_user.append(g) 

hasil = forward_chaining(aturan, fakta_user)
diagnosa = [f for f in hasil if f.startswith("DIAGNOSA")]

if diagnosa:
    print("\nHasil pemeriksaan:")
    for d in diagnosa:
        print(" -", d)
else:
    print("\nBelum bisa disimpulkan -- bawa ke teknisi ya!")

=== SISTEM PAKAR DIAGNOSA LAPTOP ===
[*] Fakta awal yang diketahui: []
--- Iterasi ke-1 ---
  (Tidak ada aturan baru yang terpicu)

Belum bisa disimpulkan -- bawa ke teknisi ya!


Pengembangan 4

In [ ]:
def forward_chaining(aturan, fakta_awal):
    fakta = list(fakta_awal)
    ada_baru = True
    while ada_baru:
        ada_baru = False
        for r in aturan:
            if all(k in fakta for k in r["jika"]) and r["maka"] not in fakta:
                fakta.append(r["maka"])
                ada_baru = True
    return fakta

print("=== SISTEM PAKAR DIAGNOSA LAPTOP ===")
print("Pilih nomor gejala yang dialami (pisahkan dengan koma, contoh: 1, 2, 4):")
for i, g in enumerate(gejala_dikenal, start=1):
    print(f"[{i:2d}] {g}")

input_raw = input("\nNomor pilihan Anda: ").split(",")

fakta_user = []
for x in input_raw:
    x_bersih = x.strip()
    if x_bersih.isdigit():
        idx = int(x_bersih) - 1
        if 0 <= idx < len(gejala_dikenal):
            gejala = gejala_dikenal[idx]
            if gejala not in fakta_user:
                fakta_user.append(gejala)

print("\n--- GEJALA YANG ANDA PILIH ---")
if fakta_user:
    for idx, f in enumerate(fakta_user, start=1):
        print(f"{idx}. {f}")
else:
    print("Tidak ada nomor gejala valid yang dimasukkan.")

hasil = forward_chaining(aturan, fakta_user)
diagnosa = [f for f in hasil if f.startswith("DIAGNOSA")]

print("\n--- HASIL PEMERIKSAAN ---")
if diagnosa:
    for d in diagnosa:##### Latihan 1 — Perluas basis pengetahuan laptop: tambahkan minimal 3 aturan baru (mis. gejala baterai cepat habis, keyboard tidak berfungsi, muncul bunyi bip). Minimal satu aturan harus berantai (kesimpulannya dipakai aturan lain). Uji dengan forward chaining.
        print(f"[!] {d}")
else:
    print("Gejala tidak cocok dengan basis aturan -- bawa ke teknisi ya!")

=== SISTEM PAKAR DIAGNOSA LAPTOP ===
Pilih nomor gejala yang dialami (pisahkan dengan koma, contoh: 1, 2, 4):
[ 1] mati total
[ 2] lampu indikator mati
[ 3] baterai kembung
[ 4] charger tidak hangat
[ 5] menyala
[ 6] layar gelap
[ 7] normal di monitor eksternal
[ 8] sangat lambat
[ 9] badan laptop panas
[10] hang saat banyak aplikasi

--- GEJALA YANG ANDA PILIH ---
Tidak ada nomor gejala valid yang dimasukkan.

--- HASIL PEMERIKSAAN ---
Gejala tidak cocok dengan basis aturan -- bawa ke teknisi ya!


#### Kegiatan 5 — ✍ LATIHAN MANDIRI

##### Latihan 1 — Perluas basis pengetahuan laptop: tambahkan minimal 3 aturan baru (mis. gejala baterai cepat habis, keyboard tidak berfungsi, muncul bunyi bip). Minimal satu aturan harus berantai (kesimpulannya dipakai aturan lain). Uji dengan forward chaining.

In [22]:
# 1. Aturan Baru
aturan_baru = [
    {
        "jika": ["menyala", "bunyi bip berulang"],
        "maka": "masalah POST/booting",
    },
    {
        "jika": ["masalah POST/booting", "layar tetap gelap"],
        "maka": "DIAGNOSA: modul RAM bermasalah atau kotor, bersihkan slot RAM",
    },
    {
        "jika": ["menyala", "baterai cepat habis"],
        "maka": "DIAGNOSA: kapasitas baterai drop/bocor, ganti baterai baru",
    },
    {
        "jika": ["menyala", "sebagian tombol keyboard tidak merespons"],
        "maka": "DIAGNOSA: membran keyboard rusak, lakukan penggantian keyboard",
    },
]


# 2. Mesin Inferensi Forward Chaining
def forward_chaining(aturan, fakta_awal):
    fakta = list(fakta_awal)
    ada_baru = True

    while ada_baru:
        ada_baru = False
        for r in aturan:
            terpenuhi = all(k in fakta for k in r["jika"])
            if terpenuhi and r["maka"] not in fakta:
                fakta.append(r["maka"])
                print(f"  [Terpicu] {r['jika']} -> '{r['maka']}'")
                ada_baru = True
    return fakta


# 3. Pengujian Kasus
print("=" * 60)
print("PENGUJIAN 1: Kasus Berantai (Bunyi Bip + Layar Gelap)")
gejala_uji_1 = ["menyala", "bunyi bip berulang", "layar tetap gelap"]
print("Gejala awal :", gejala_uji_1)

hasil_1 = forward_chaining(aturan_baru, gejala_uji_1)
diagnosa_1 = [f for f in hasil_1 if f.startswith("DIAGNOSA")]
print("Diagnosa    :", diagnosa_1)

print("\n" + "=" * 60)
print("PENGUJIAN 2: Kasus Baterai Drop")
gejala_uji_2 = ["menyala", "baterai cepat habis"]
print("Gejala awal :", gejala_uji_2)

hasil_2 = forward_chaining(aturan_baru, gejala_uji_2)
diagnosa_2 = [f for f in hasil_2 if f.startswith("DIAGNOSA")]
print("Diagnosa    :", diagnosa_2)

print("\n" + "=" * 60)
print("PENGUJIAN 3: Kasus Keyboard Rusak")
gejala_uji_3 = ["menyala", "sebagian tombol keyboard tidak merespons"]
print("Gejala awal :", gejala_uji_3)

hasil_3 = forward_chaining(aturan_baru, gejala_uji_3)
diagnosa_3 = [f for f in hasil_3 if f.startswith("DIAGNOSA")]
print("Diagnosa    :", diagnosa_3)

PENGUJIAN 1: Kasus Berantai (Bunyi Bip + Layar Gelap)
Gejala awal : ['menyala', 'bunyi bip berulang', 'layar tetap gelap']
  [Terpicu] ['menyala', 'bunyi bip berulang'] -> 'masalah POST/booting'
  [Terpicu] ['masalah POST/booting', 'layar tetap gelap'] -> 'DIAGNOSA: modul RAM bermasalah atau kotor, bersihkan slot RAM'
Diagnosa    : ['DIAGNOSA: modul RAM bermasalah atau kotor, bersihkan slot RAM']

PENGUJIAN 2: Kasus Baterai Drop
Gejala awal : ['menyala', 'baterai cepat habis']
  [Terpicu] ['menyala', 'baterai cepat habis'] -> 'DIAGNOSA: kapasitas baterai drop/bocor, ganti baterai baru'
Diagnosa    : ['DIAGNOSA: kapasitas baterai drop/bocor, ganti baterai baru']

PENGUJIAN 3: Kasus Keyboard Rusak
Gejala awal : ['menyala', 'sebagian tombol keyboard tidak merespons']
  [Terpicu] ['menyala', 'sebagian tombol keyboard tidak merespons'] -> 'DIAGNOSA: membran keyboard rusak, lakukan penggantian keyboard'
Diagnosa    : ['DIAGNOSA: membran keyboard rusak, lakukan penggantian keyboard']


##### Latihan 2 — Bangun sistem pakar domain pilihanmu (mis. diagnosa motor tidak mau hidup, troubleshooting WiFi, penyakit tanaman): minimal 6 aturan dengan minimal 2 rantai, lalu jalankan versi interaktifnya.

In [23]:
# 1. Basis Pengetahuan (Knowledge Base) Troubleshooting Wi-Fi
aturan = [
    # Rantai 1: Inferensi perantara masalah DNS
    {
        "jika": ["bisa ping gateway", "gagal buka web domain"],
        "maka": "koneksi lokal normal tapi domain gagal",
    },
    {
        "jika": [
            "koneksi lokal normal tapi domain gagal",
            "bisa ping 8.8.8.8 lewat IP",
        ],
        "maka": "DIAGNOSA: DNS bermasalah (flush DNS atau ganti ke 1.1.1.1 / 8.8.8.8)",
    },
    # Rantai 2: Inferensi perantara masalah IP Address
    {
        "jika": ["sinyal wifi penuh", "status no internet access"],
        "maka": "masalah alokasi IP",
    },
    {
        "jika": ["masalah alokasi IP", "ip berawalan 169.254"],
        "maka": "DIAGNOSA: Gagal dapat DHCP (restart router atau release/renew IP)",
    },
    {
        "jika": ["lampu LOS router merah"],
        "maka": "DIAGNOSA: Kabel fiber optik putus atau redaman tinggi, lapor teknisi ISP",
    },
    {
        "jika": ["nama wifi tidak muncul", "perangkat lain bisa melihat"],
        "maka": "DIAGNOSA: Driver adapter Wi-Fi bermasalah atau band frekuensi tidak cocok (2.4GHz vs 5GHz)",
    },
    {
        "jika": ["status autentikasi gagal", "password sudah benar"],
        "maka": "DIAGNOSA: Konflik security mode router (WPA2 vs WPA3), lupakan jaringan lalu sambung ulang",
    },
]

# 2. Daftar Gejala yang Bisa Dipilih Pengguna
gejala_dikenal = [
    "bisa ping gateway",
    "gagal buka web domain",
    "bisa ping 8.8.8.8 lewat IP",
    "sinyal wifi penuh",
    "status no internet access",
    "ip berawalan 169.254",
    "lampu LOS router merah",
    "nama wifi tidak muncul",
    "perangkat lain bisa melihat",
    "status autentikasi gagal",
    "password sudah benar",
]

# 3. Mesin Inferensi Forward Chaining
def forward_chaining(aturan, fakta_awal):
    fakta = list(fakta_awal)
    ada_baru = True

    while ada_baru:
        ada_baru = False
        for r in aturan:
            terpenuhi = all(k in fakta for k in r["jika"])
            if terpenuhi and r["maka"] not in fakta:
                fakta.append(r["maka"])
                ada_baru = True

    return fakta

# 4. Antarmuka Interaktif CLI
print("=== SISTEM PAKAR TROUBLESHOOTING WI-FI & INTERNET ===")
print("Pilih nomor masalah/gejala yang terjadi (pisahkan dengan koma, contoh: 1, 2, 3):")
for i, g in enumerate(gejala_dikenal, start=1):
    print(f"[{i:2d}] {g}")

input_raw = input("\nNomor gejala: ").split(",")

fakta_user = []
for x in input_raw:
    x_bersih = x.strip()
    if x_bersih.isdigit():
        idx = int(x_bersih) - 1
        if 0 <= idx < len(gejala_dikenal):
            gejala = gejala_dikenal[idx]
            if gejala not in fakta_user:
                fakta_user.append(gejala)

# Tampilkan ringkasan gejala
print("\n--- GEJALA TERDETEKSI ---")
if fakta_user:
    for idx, f in enumerate(fakta_user, start=1):
        print(f"{idx}. {f}")
else:
    print("(Tidak ada pilihan valid yang dimasukkan)")

# Eksekusi inferensi
hasil = forward_chaining(aturan, fakta_user)
diagnosa = [f for f in hasil if f.startswith("DIAGNOSA")]

print("\n--- REKOMENDASI & DIAGNOSA AKHIR ---")
if diagnosa:
    for d in diagnosa:
        print(f"[!] {d}")
else:
    print("Gejala belum cukup spesifik untuk menentukan solusi pasti.")

=== SISTEM PAKAR TROUBLESHOOTING WI-FI & INTERNET ===
Pilih nomor masalah/gejala yang terjadi (pisahkan dengan koma, contoh: 1, 2, 3):
[ 1] bisa ping gateway
[ 2] gagal buka web domain
[ 3] bisa ping 8.8.8.8 lewat IP
[ 4] sinyal wifi penuh
[ 5] status no internet access
[ 6] ip berawalan 169.254
[ 7] lampu LOS router merah
[ 8] nama wifi tidak muncul
[ 9] perangkat lain bisa melihat
[10] status autentikasi gagal
[11] password sudah benar

--- GEJALA TERDETEKSI ---
1. gagal buka web domain

--- REKOMENDASI & DIAGNOSA AKHIR ---
Gejala belum cukup spesifik untuk menentukan solusi pasti.


##### Latihan 3 — Analisis (markdown): kapan forward lebih cocok daripada backward chaining pada kasusmu? Sebutkan juga 2 keterbatasan sistem pakar dibanding manusia/machine learning (minimal 4 kalimat).

##### 1. Kapan Forward Chaining Lebih Cocok Dibanding Backward Chaining?

Pada kasus **troubleshooting Wi-Fi/Internet**, **Forward Chaining** jauh lebih cocok digunakan ketika pengguna datang dengan banyak gejala awal tanpa tahu apa akar masalahnya**. 

* **Alasannya:** Pengguna biasanya hanya tahu kondisi riil di lapangan, misalnya sinyal penuh, lampu router merah, atau web tidak mau terbuka. Melalui forward chaining (*data-driven*), sistem mengumpulkan semua keluhan di awal, merangkainya satu per satu seperti detektif yang menyusun kepingan petunjuk, lalu menyimpulkan diagnosa yang paling pas secara alami.
* **Bandingannya:** Jika memakai **Backward Chaining** (*goal-driven*), sistem harus menebak satu diagnosa di awal (misalnya curiga *"kabel fiber optik putus"*), baru kemudian mencecar pengguna dengan pertanyaan untuk membuktikannya. Cara ini kurang efisien jika kemungkinan kerusakannya sangat banyak, karena pengguna bisa ditanya pertanyaan yang sebenarnya tidak relevan dengan kondisi aslinya.

---

#### 2. Keterbatasan Sistem Pakar Dibanding Manusia & Machine Learning

Meskipun sistem pakar berbasis aturan (*rule-based*) sangat transparan dan mudah ditelusuri alurnya, metode ini memiliki beberapa keterbatasan mendasar:

1. Tidak Bisa Belajar Sendiri
   Sistem pakar sangat bergantung pada aturan baku yang dimasukkan manusia secara manual (*hardcoded*). Jika ada pola kerusakan jaringan baru yang belum terdaftar di basis aturan, sistem akan langsung buntu dan gagal memberikan diagnosa. Berbeda dengan Machine Learning yang otomatis bertambah pintar dengan membaca data riwayat jaringan, sistem pakar harus menunggu *programmer* atau pakar memperbarui kodenya secara manual.

2. Tidak Punya Intuisi dan Kesulitan Mengolah Gejala yang Ambigu:
   Teknisi manusia bisa memadukan insting lapangan, pengalaman bertahun-tahun, serta melihat konteks lingkungan saat mendiagnosa masalah yang rancu misalnya koneksi kadang putus hanya saat hujan. Sebaliknya, sistem pakar hanya melihat logika hitam-putih (*Benar* atau *Salah*). Jika pengguna ragu-ragu atau salah menjawab satu gejala saja, seluruh alur penalaran sistem pakar bisa langsung meleset tanpa kemampuan menoleransi ambiguitas seperti manusia atau algoritma probabilitas modern.

## Buku Ajar - Bab 04 Sistem Pakar

##### **A. Pilihan ganda**

#### 1. Komponen yang memungkinkan basis pengetahuan diganti tanpa mengubah program adalah pemisahan basis pengetahuan dari…
* [ ] (a) antarmuka
* [x] **(b) mesin inferensi**
* [ ] (c) memori kerja
* [ ] (d) pengguna

#### 2. Fakta baru hasil penalaran disimpan di…
* [ ] (a) basis pengetahuan
* [x] **(b) memori kerja**
* [ ] (c) fasilitas penjelasan
* [ ] (d) basis data pakar

#### 3. Penalaran yang berangkat dari dugaan lalu mencari bukti disebut…
* [ ] (a) forward chaining
* [x] **(b) backward chaining**
* [ ] (c) modus ponens
* [ ] (d) pewarisan

#### 4. Sistem monitoring suhu pabrik yang menerima ratusan pembacaan sensor per detik paling cocok memakai…
* [ ] (a) backward chaining
* [x] **(b) forward chaining**
* [ ] (c) tabel kebenaran
* [ ] (d) frame

#### 5. CF aturan 0,9; CF kondisi-kondisinya 0,7 dan 0,4. CF kesimpulan = …
* [ ] (a) 0,9
* [ ] (b) 0,63
* [x] **(c) 0,36**
* [ ] (d) 0,28

##### **B. Esai**

#### 1. Rancang basis pengetahuan 8 aturan untuk domain pilihanmu (mis. motor tak mau hidup, printer bermasalah) dengan minimal dua rantai; tandai mana kesimpulan antara dan mana diagnosa akhir.
- **R1:** JIKA starter hening DAN klakson mati MAKA masalah kelistrikan (Antara)
- **R2:** JIKA masalah kelistrikan DAN aki drop MAKA DIAGNOSA: Aki Soak (Akhir)
- **R3:** JIKA starter hening DAN klakson bunyi MAKA masalah sakelar (Antara)
- **R4:** JIKA masalah sakelar DAN standar samping turun MAKA DIAGNOSA: Standar Samping Aktif (Akhir)
- **R5:** JIKA starter muter DAN mesin gamau hidup MAKA masalah pengapian/bensin (Antara)
- **R6:** JIKA masalah pengapian/bensin DAN busi mati MAKA DIAGNOSA: Ganti Busi (Akhir)
- **R7:** JIKA masalah pengapian/bensin DAN busi nyala MAKA masalah bensin (Antara)
- **R8:** JIKA masalah bensin DAN fuel pump hening MAKA DIAGNOSA: Fuel Pump Rusak (Akhir)

#### 2. Dengan basis aturanmu pada soal 1, buat trace table forward chaining untuk satu set fakta pilihan — tunjukkan putaran demi putaran sampai berhenti.
- Fakta awal: starter muter, mesin gamau hidup, busi nyala, fuel pump hening.
- Putaran 1:
  - R5 jalan -> fakta baru: masalah pengapian/bensin
  - R7 jalan -> fakta baru: masalah bensin
  - R8 jalan -> DIAGNOSA: Fuel Pump Rusak
- Putaran 2: Tidak ada aturan baru. Selesai.

#### 3. Masih dengan basis yang sama, buktikan satu diagnosa lewat backward chaining; tuliskan pohon dugaan–sub-dugaannya.
- Dugaan diuji: Ganti Busi (R6).
- Cek syarat R6:
  - Syarat 1 (masalah pengapian/bensin dari R5) -> Terbukti.
  - Syarat 2 (busi mati) -> Gagal, faktanya busi nyala.
- Hasil: Dugaan Ganti Busi Ditolak.

#### 4. Sebuah rumah sakit meminta “sistem pakar yang mendiagnosis segala penyakit”. Evaluasi permintaan ini dengan tiga syarat kelayakan Subbab 4.1, lalu usulkan lingkup yang lebih realistis.
Permintaan ini tidak layak karena:
- Domain terlalu luas (sistem pakar butuh lingkup sempit).
- Gejala ribuan penyakit saling tumpang-tindih dan sulit dibuat if-then kaku.
- Tidak ada satu orang dokter pun yang ahli di semua penyakit.
- Saran lingkup realistis: Sistem Skrining Awal Demam Berdarah dan Tifus pada Anak.

#### 5. Mengapa fasilitas penjelasan (“mengapa Anda menyimpulkan itu?”) penting bagi kepercayaan pengguna — dan bandingkan dengan model machine learning yang sulit menjelaskan dirinya (pratinjau Bab 7).
- Fasilitas penjelasan penting supaya pengguna tahu alasan sistem mengambil keputusan, jadi hasilnya bisa dipercaya dan dicek ulang.
- Bedanya dengan Machine Learning: Sistem pakar transparan (White Box) karena alur if-then-nya kelihatan jelas, sedangkan Machine Learning cenderung seperti kotak hitam (Black Box) yang sulit menjelaskan alasan di balik jawabannya.